In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

/home/ailab/miniconda3/envs/GNNTrain/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================
# 1. 加载数据并构建 AnnData 对象（可选，此处直接使用 pandas）
# ============================================
print("Loading data...")
data_path = "./data/xenium_for_gnn.csv"
df = pd.read_csv(data_path)

# 查看列名
print("Columns:", df.columns.tolist())

# 确定列名（根据实际导出调整）
# 假设 cell_id 列名为 'cell_id'
# 坐标列名：通常为 'x_centroid', 'y_centroid'，如果不同请修改
# PCA 特征列：假设为 'PC_1' 到 'PC_50'
# 标签列：'predicted.id'（之前导出的预测细胞类型）

cell_id_col = 'cell_id'
x_col = 'x_centroid' if 'x_centroid' in df.columns else 'x'   # 请根据实际列名修改
y_col = 'y_centroid' if 'y_centroid' in df.columns else 'y'
label_col = 'predicted.id'   # 或 'predicted_cell_type'，请核对

# 提取特征：PCA 列（假设以 'PC_' 开头）
feature_cols = [c for c in df.columns if c.startswith('PC_')]
print(f"Found {len(feature_cols)} PCA features.")
if len(feature_cols) == 0:
    # 如果没有 PCA 列，可能需要使用其他特征，此处给出错误提示
    raise ValueError("No PCA features found. Please check exported data.")

# 提取空间坐标
coords = df[[x_col, y_col]].values

# 提取标签并转换为整数编码
labels_original = df[label_col].values
unique_labels = np.unique(labels_original)
label_to_int = {lbl: i for i, lbl in enumerate(unique_labels)}
labels = np.array([label_to_int[lbl] for lbl in labels_original])
num_classes = len(unique_labels)

# 提取特征矩阵
X = df[feature_cols].values

print(f"Data shape: {X.shape}")
print(f"Number of cells: {X.shape[0]}")
print(f"Number of classes: {num_classes}")
print(f"Classes: {unique_labels}")


Loading data...
Columns: ['cell_id', 'x_centroid', 'y_centroid', 'predicted_cell_type', 'PC_1', 'PC_2', 'PC_3', 'PC_4', 'PC_5', 'PC_6', 'PC_7', 'PC_8', 'PC_9', 'PC_10', 'PC_11', 'PC_12', 'PC_13', 'PC_14', 'PC_15', 'PC_16', 'PC_17', 'PC_18', 'PC_19', 'PC_20', 'PC_21', 'PC_22', 'PC_23', 'PC_24', 'PC_25', 'PC_26', 'PC_27', 'PC_28', 'PC_29', 'PC_30', 'PC_31', 'PC_32', 'PC_33', 'PC_34', 'PC_35', 'PC_36', 'PC_37', 'PC_38', 'PC_39', 'PC_40', 'PC_41', 'PC_42', 'PC_43', 'PC_44', 'PC_45', 'PC_46', 'PC_47', 'PC_48', 'PC_49', 'PC_50']
Found 50 PCA features.


KeyError: 'predicted.id'

In [ ]:
# ============================================
# 2. 构建 kNN 空间图
# ============================================
print("\nBuilding kNN graph...")
k = 30  # 邻居数，可根据细胞密度调整
adj = kneighbors_graph(coords, n_neighbors=k, mode='connectivity', include_self=False)
edge_index = torch.tensor(np.array(adj.nonzero()), dtype=torch.long)

# 转换为 PyG Data 对象
data = Data(x=torch.tensor(X, dtype=torch.float),
            edge_index=edge_index,
            y=torch.tensor(labels, dtype=torch.long))

# 添加空间坐标作为可选信息（用于可视化）
data.pos = torch.tensor(coords, dtype=torch.float)

print(f"Graph built: {edge_index.shape[1]} edges")


In [ ]:
# ============================================
# 3. 划分训练/验证/测试集
# ============================================
print("\nSplitting dataset...")
# 按细胞划分，确保空间连续性不被破坏？此处简单随机划分
train_idx, test_idx = train_test_split(np.arange(X.shape[0]), test_size=0.2, random_state=42, stratify=labels)
train_idx, val_idx = train_test_split(train_idx, test_size=0.2, random_state=42, stratify=labels[train_idx])

# 创建 mask
train_mask = torch.zeros(X.shape[0], dtype=torch.bool)
val_mask = torch.zeros(X.shape[0], dtype=torch.bool)
test_mask = torch.zeros(X.shape[0], dtype=torch.bool)
train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

data.train_mask = train_mask
data.val_mask = val_mask
data.test_mask = test_mask

print(f"Train: {train_mask.sum().item()}, Val: {val_mask.sum().item()}, Test: {test_mask.sum().item()}")

In [ ]:
# ============================================
# 4. 定义 GNN 模型
# ============================================
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=4, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=dropout)
        # 注意：GATConv 输出维度 = hidden_channels * heads
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=dropout)
        self.dropout = dropout

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return F.log_softmax(x, dim=1)


In [ ]:
# ============================================
# 5. 训练函数
# ============================================
def train(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    out = model(data)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out = model(data)
        pred = out.argmax(dim=1)
        correct = (pred[mask] == data.y[mask]).sum()
        acc = int(correct) / int(mask.sum())
        return acc, pred[mask].cpu().numpy(), data.y[mask].cpu().numpy()

In [ ]:
# ============================================
# 6. 主训练循环（以 GCN 为例）
# ============================================
def run_experiment(model_class, name, hidden=64, lr=0.01, epochs=200, patience=10):
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    data_device = data.to(device)
    model = model_class(in_channels=X.shape[1], hidden_channels=hidden, out_channels=num_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    criterion = torch.nn.NLLLoss()

    best_val_acc = 0
    best_epoch = 0
    wait = 0
    train_losses = []
    val_accs = []

    for epoch in range(1, epochs+1):
        loss = train(model, data_device, optimizer, criterion)
        train_acc, _, _ = evaluate(model, data_device, data_device.train_mask)
        val_acc, _, _ = evaluate(model, data_device, data_device.val_mask)
        train_losses.append(loss)
        val_accs.append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            wait = 0
            # 保存最佳模型
            torch.save(model.state_dict(), f'best_{name}.pt')
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 20 == 0:
            print(f'Epoch {epoch:03d}, Loss: {loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')

    # 加载最佳模型并在测试集上评估
    model.load_state_dict(torch.load(f'best_{name}.pt'))
    test_acc, test_pred, test_true = evaluate(model, data_device, data_device.test_mask)
    print(f"\n{name} Test Accuracy: {test_acc:.4f}")

    # 计算宏平均 F1
    test_f1 = f1_score(test_true, test_pred, average='macro')
    print(f"{name} Macro F1: {test_f1:.4f}")

    # 详细分类报告
    print("\nClassification Report:")
    print(classification_report(test_true, test_pred, target_names=unique_labels))

    # 混淆矩阵
    cm = confusion_matrix(test_true, test_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=unique_labels, yticklabels=unique_labels)
    plt.title(f'{name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.tight_layout()
    plt.savefig(f'confusion_matrix_{name}.png', dpi=300)
    plt.show()

    # 可视化预测结果在空间上的分布（采样部分细胞）
    # 注意：数据量很大，可随机采样 10000 点或降采样显示
    np.random.seed(42)
    idx_sample = np.random.choice(data_device.num_nodes, size=min(10000, data_device.num_nodes), replace=False)
    sample_coords = data_device.pos[idx_sample].cpu().numpy()
    sample_pred = pred[idx_sample].cpu().numpy()
    sample_true = data_device.y[idx_sample].cpu().numpy()

    # 将预测标签映射回原始名称
    pred_names = [unique_labels[i] for i in sample_pred]
    true_names = [unique_labels[i] for i in sample_true]

    # 空间散点图
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sc1 = axes[0].scatter(sample_coords[:,0], sample_coords[:,1], c=sample_pred, cmap='tab20', s=1, alpha=0.6)
    axes[0].set_title(f'{name} Predicted Cell Types (sample)')
    axes[0].set_xlabel('X')
    axes[0].set_ylabel('Y')
    plt.colorbar(sc1, ax=axes[0], ticks=range(num_classes), label='Cell Type ID')

    sc2 = axes[1].scatter(sample_coords[:,0], sample_coords[:,1], c=sample_true, cmap='tab20', s=1, alpha=0.6)
    axes[1].set_title('True Labels (sample)')
    axes[1].set_xlabel('X')
    axes[1].set_ylabel('Y')
    plt.colorbar(sc2, ax=axes[1], ticks=range(num_classes), label='Cell Type ID')
    plt.tight_layout()
    plt.savefig(f'spatial_predictions_{name}.png', dpi=300)
    plt.show()

    return test_acc, test_f1

In [ ]:
# ============================================
# 7. 运行所有模型对比
# ============================================
results = {}

# GCN
acc, f1 = run_experiment(GCN, "GCN", hidden=64)
results['GCN'] = (acc, f1)

# GraphSAGE
acc, f1 = run_experiment(GraphSAGE, "GraphSAGE", hidden=64)
results['GraphSAGE'] = (acc, f1)

# GAT（可能需要更长时间）
acc, f1 = run_experiment(GAT, "GAT", hidden=64, heads=4)
results['GAT'] = (acc, f1)

# 打印汇总结果
print("\n" + "="*50)
print("Summary of Results:")
print("Model\t\tTest Accuracy\tMacro F1")
for name, (acc, f1) in results.items():
    print(f"{name}\t\t{acc:.4f}\t\t{f1:.4f}")

# ============================================
# 8. 基线：MLP（不使用图结构）
# ============================================
print("\n" + "="*50)
print("Training MLP baseline (no graph)...")
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, f1_score

# 准备训练数据
X_train = X[train_idx]
y_train = labels[train_idx]
X_val = X[val_idx]
y_val = labels[val_idx]
X_test = X[test_idx]
y_test = labels[test_idx]

mlp = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=300, random_state=42, early_stopping=True, validation_fraction=0.1)
mlp.fit(X_train, y_train)

y_pred = mlp.predict(X_test)
mlp_acc = accuracy_score(y_test, y_pred)
mlp_f1 = f1_score(y_test, y_pred, average='macro')
print(f"MLP Test Accuracy: {mlp_acc:.4f}")
print(f"MLP Macro F1: {mlp_f1:.4f}")
print("\nMLP Classification Report:")
print(classification_report(y_test, y_pred, target_names=unique_labels))

# 添加 MLP 结果到汇总
results['MLP'] = (mlp_acc, mlp_f1)

print("\n" + "="*50)
print("Final Comparison:")
print("Model\t\tTest Accuracy\tMacro F1")
for name, (acc, f1) in results.items():
    print(f"{name}\t\t{acc:.4f}\t\t{f1:.4f}")

# 可选：绘制结果柱状图
models = list(results.keys())
accs = [results[m][0] for m in models]
f1s = [results[m][1] for m in models]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
rects1 = ax.bar(x - width/2, accs, width, label='Accuracy')
rects2 = ax.bar(x + width/2, f1s, width, label='Macro F1')
ax.set_ylabel('Score')
ax.set_title('Model Comparison')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.bar_label(rects1, fmt='%.3f', padding=3)
ax.bar_label(rects2, fmt='%.3f', padding=3)
fig.tight_layout()
plt.savefig('model_comparison.png', dpi=300)
plt.show()